In [ ]:
# AB测试统计功效模拟
# 场景：定价策略师想测试"满30减5"vs"满25减3"哪个GMV更高
# 已知：历史GMV均值=100元，标准差=30元
# 问题：每组需要多少样本才能检测到5元的GMV差异？

import numpy as np
import scipy.stats as st
import matplotlib.pyplot as plt

# ============================================
# Part 1: 理解统计功效 — 通俗模拟
# ============================================
np.random.seed(42)

def simulate_ab_test(n_per_group, true_effect, std, alpha=0.05):
    """
    模拟一次AB测试
    n_per_group: 每组样本量
    true_effect: B组相比A组的真实效果（效应量）
    std: 标准差
    alpha: 显著性水平
    """
    # 生成两组数据：A组均值=100，B组均值=100+effect
    group_a = np.random.normal(100, std, n_per_group)
    group_b = np.random.normal(100 + true_effect, std, n_per_group)
    
    # 独立样本t检验
    t_stat, p_value = st.ttest_ind(group_b, group_a)
    
    return p_value < alpha  # True=检测到显著差异

def calc_statistical_power(n_per_group, true_effect, std, n_simulations=1000):
    """通过多次模拟计算统计功效"""
    significant_count = sum(
        simulate_ab_test(n_per_group, true_effect, std)
        for _ in range(n_simulations)
    )
    return significant_count / n_simulations

# ----- 演示1：固定效应量，看样本量如何影响功效 -----
effect = 5   # B组比A组高5元
std_dev = 30  # 标准差30元
sample_sizes = [50, 100, 200, 500, 1000, 2000]

powers = [calc_statistical_power(n, effect, std_dev) for n in sample_sizes]

print("=" * 50)
print("场景：B组真实GMV比A组高5元，标准差=30元")
print("不同样本量下的统计功效：")
print("=" * 50)
for n, p in zip(sample_sizes, powers):
    status = "✓ 够用" if p >= 0.8 else "✗ 不足"
    print(f"  每组 {n:>5} 人 → 功效 = {p:.1%}  {status}")

# ----- 演示2：固定样本量，看效应量如何影响功效 -----
n_fixed = 500
effects = [1, 2, 3, 5, 8, 10]

powers_by_effect = [calc_statistical_power(n_fixed, e, std_dev) for e in effects]

print("\n" + "=" * 50)
print("场景：固定每组500样本，标准差=30元")
print("不同效应量下的统计功效：")
print("=" * 50)
for e, p in zip(effects, powers_by_effect):
    status = "✓ 能检测到" if p >= 0.8 else "✗ 检测不到"
    print(f"  效应量 {e:>2} 元 → 功效 = {p:.1%}  {status}")

# ----- 演示3：可视化 -----
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(sample_sizes, powers, 'o-', linewidth=2, markersize=8)
axes[0].axhline(0.8, color='red', linestyle='--', label='80%功效（行业标准）')
axes[0].set_xlabel('每组样本量')
axes[0].set_ylabel('统计功效')
axes[0].set_title('样本量 vs 统计功效（效应量固定=5元）')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(effects, powers_by_effect, 'o-', linewidth=2, markersize=8, color='orange')
axes[1].axhline(0.8, color='red', linestyle='--', label='80%功效（行业标准）')
axes[1].set_xlabel('效应量（元）')
axes[1].set_ylabel('统计功效')
axes[1].set_title('效应量 vs 统计功效（每组固定500样本）')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# Part 2: MDE与样本量计算 — 公式推导 + 模拟验证
# ============================================
from scipy.stats import norm

def calc_mde(std, n_per_group, alpha=0.05, power=0.8):
    """
    计算MDE（最小可检测效应）
    公式：MDE = (Z_{1-alpha/2} + Z_{power}) * std * sqrt(2/n)
    其中 Z_{1-alpha/2} 是双尾检验的临界值，Z_{power} 是功效对应的Z值
    """
    z_alpha = norm.ppf(1 - alpha / 2)  # 双尾，alpha=0.05时约为1.96
    z_power = norm.ppf(power)          # power=0.8时约为0.84
    
    mde = (z_alpha + z_power) * std * np.sqrt(2 / n_per_group)
    return mde

def calc_sample_size(std, mde, alpha=0.05, power=0.8):
    """
    反算所需样本量
    公式：n = 2 * ((Z_{alpha/2} + Z_{beta}) * std / MDE)^2
    """
    z_alpha = norm.ppf(1 - alpha / 2)
    z_beta = norm.ppf(power)
    
    n = 2 * ((z_alpha + z_beta) * std / mde) ** 2
    return int(np.ceil(n))

# ----- 演示：不同的样本量下能检测到的最小效果 -----
print("=" * 50)
print("MDE计算：每组不同样本量能检测到的最小GMV差异")
print("（alpha=0.05, power=0.8, std=30元）")
print("=" * 50)

for n in [100, 200, 500, 1000, 2000, 5000]:
    mde = calc_mde(30, n)
    print(f"  每组 {n:>5} 人 → 能检测到的最小差异 = {mde:.1f} 元")

# ----- 演示：如果要检测5元差异，需要多少样本？ -----
print("\n" + "=" * 50)
print("样本量计算：要检测到X元的GMV差异，每组需要多少人？")
print("（alpha=0.05, power=0.8, std=30元）")
print("=" * 50)

for target_mde in [10, 8, 5, 3, 2, 1]:
    n_needed = calc_sample_size(30, target_mde)
    print(f"  要检测 {target_mde} 元差异 → 每组需要 {n_needed:>6} 人")

# ----- 定价场景的实际决策推理 -----
print("\n" + "=" * 50)
print("定价决策推理：")
print("=" * 50)
print("假设DAU=50,000，能做10%流量实验（每组2,500人）")
n_real = 2500
mde_real = calc_mde(30, n_real)
print(f"→ 每组2,500人时，MDE = {mde_real:.1f}元")
print(f"→ 如果你的优惠券方案预期只提升2元GMV，当前流量不够，需要扩大实验比例或延长实验时间")
print(f"→ 这是定价分析师做实验设计时最重要的判断：样本量够不够")